# Bài học 9: Agent với tools

## Tại sao agent cần tools?

Ở bài trước, agent của chúng ta chỉ có thể **suy nghĩ** và **trả lời** dựa trên kiến thức sẵn có.

Nhưng nếu bạn hỏi: *"Xu hướng SEO mới nhất năm 2026 là gì?"* — Agent không biết vì kiến thức của nó có giới hạn thời gian.

Ở Bài học 5, ta đã biết rằng LLM được huấn luyện trên dữ liệu đến một ngày nhất định — sau đó, chúng hoàn toàn mù thông tin. **Tools** chính là cách giải quyết vấn đề này. Chúng cho phép agent **làm** được nhiều hơn:
- Tìm kiếm trên web
- Truy vấn cơ sở dữ liệu
- Tìm hình ảnh
- Và nhiều thứ khác...

Giống như nhân viên có thể đọc tin tức và tìm kiếm trên Google, agent có tools cũng có thể tìm kiếm thông tin theo thời gian thực.

> **Chi phí:** ~$0.02-0.10 mỗi ô (Sonnet gọi kèm web search). Chạy mỗi ô một lần để tiết kiệm chi phí.
>
> **Yêu cầu:** `DATA_FOR_SEO_API_KEY` đã được cài trong file `.env`. Nếu chưa có, các lệnh search sẽ lỗi nhưng bạn vẫn có thể đọc code để hiểu cách hoạt động.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## DataForSEO — Công cụ tìm kiếm chuyên nghiệp

**DataForSEO** là API tìm kiếm web được sử dụng trong sản phẩm của chúng ta. Nó truy vấn Google và trả về kết quả có cấu trúc.

Khác với các công cụ miễn phí (dễ bị giới hạn tốc độ khi dùng nhiều), DataForSEO là **API trả phí** được các công cụ SEO chuyên nghiệp trên thế giới sử dụng.

Sản phẩm của chúng ta gói nó trong một **Toolkit** — một class mà agent có thể gọi:

```python
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

creds = get_dataforseo_credentials()  # Đọc DATA_FOR_SEO_API_KEY từ .env
search = DataForSEOSearchTools(login=creds[0], password=creds[1])

agent = Agent(
    tools=[search],  # Trao cho agent khả năng tìm kiếm
)
```

`get_dataforseo_credentials()` giải mã API key từ `.env` thành cặp login/password. `DataForSEOSearchTools` toolkit cung cấp cho agent phương thức `web_search()` để gọi.

In [ ]:
from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

# Cài đặt search tool với credentials từ .env
creds = get_dataforseo_credentials()
if creds is None:
    print("CẢNH BÁO: DATA_FOR_SEO_API_KEY chưa được cài. Các ví dụ search sẽ không hoạt động.")
    print("Bạn vẫn có thể đọc code để hiểu cách hoạt động.")
    search_tools = []
else:
    search_tools = [DataForSEOSearchTools(login=creds[0], password=creds[1])]
    print("DataForSEO đã cấu hình xong! Search tools sẵn sàng.")

# Agent có khả năng tìm kiếm web!
agent = Agent(
    name="Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web for information and summarize the results."],
)

response = agent.run("Find the top 3 SEO trends in 2026")
print(response.content)

## Điều gì vừa xảy ra phía sau?

Khi bạn chạy đoạn code ở trên, đây là những gì diễn ra:

1. **Bạn gửi câu hỏi** — "Find the top 3 SEO trends in 2026"
2. **Agent quyết định** — "Mình cần thông tin mới, dùng web_search thôi"
3. **Agent gọi tool** — Tìm kiếm Google qua DataForSEO API
4. **Agent đọc kết quả** — Xử lý các kết quả tìm kiếm
5. **Agent tóm tắt** — Viết câu trả lời dựa trên những gì tìm được

**Điểm mấu chốt:** Bạn không cần tự viết code tìm kiếm. Agent **tự quyết định** khi nào cần dùng tools.

### Bên trong: Tool call hoạt động như thế nào

<details>
<summary>Bấm để mở rộng — phần hiểu sâu hơn, không bắt buộc</summary>

Khi agent "gọi tool," thực tế đang diễn ra một **API call** — một yêu cầu gửi qua internet. Đây là phiên bản đơn giản:

**API là gì?** Hãy tưởng tượng một nhà hàng:
- Bạn (agent) đặt **món** (request) với bồi bàn (API)
- Bếp (server) chuẩn bị món ăn (dữ liệu)
- Bồi bàn mang lại cho bạn (response)

DataForSEO chính là nhà bếp. Agent gửi yêu cầu ("tìm kiếm 'SEO trends 2026'"), DataForSEO trả về kết quả có cấu trúc.

**Hai loại request:**

| Loại | Khi nào | Ví dụ |
|------|---------|-------|
| **GET** | Lấy dữ liệu | "Cho tôi kết quả tìm kiếm cho keyword này" |
| **POST** | Gửi dữ liệu | "Đây là truy vấn tìm kiếm, xử lý và trả kết quả" |

**Mã trạng thái (status code)** cho biết chuyện gì đã xảy ra:

| Mã | Ý nghĩa | Cần làm gì |
|-----|---------|-----------|
| 200 | Thành công | Xử lý kết quả |
| 401 | API key sai | Kiểm tra file `.env` |
| 404 | Không tìm thấy | Kiểm tra URL endpoint |
| 429 | Quá nhiều request | Đợi rồi thử lại |
| 500 | Lỗi server | Thử lại sau |

**Response trả về dưới dạng JSON** — dữ liệu có cấu trúc mà Python đọc được như dictionary:
```python
{"status": 200, "results": [{"title": "SEO Guide", "url": "..."}]}
```

Bạn không cần tự viết bất kỳ thứ gì ở trên — `DataForSEOSearchTools` toolkit xử lý hết. Nhưng hiểu điều gì đang xảy ra sẽ giúp bạn debug khi có lỗi (ví dụ: "lỗi 401? Kiểm tra API key").

</details>

In [ ]:
# Ví dụ khác — nghiên cứu một chủ đề cụ thể
response = agent.run("What is SEONGON? Find information about this company.")
print(response.content)

## So sánh: Agent KHÔNG CÓ tools vs CÓ tools

Sự khác biệt:

| | Không có Tools | Có Tools |
|---|---|---|
| Kiến thức | Chỉ biết những gì được huấn luyện | Có thể tìm kiếm thông tin mới |
| Thông tin | Có thể lỗi thời/sai | Cập nhật và chính xác |
| Khả năng | Chỉ suy nghĩ | Suy nghĩ + Hành động |

Chạy code bên dưới để thấy rõ sự khác biệt.

In [ ]:
# Agent KHÔNG CÓ tools
agent_no_tools = Agent(
    name="Agent Without Tools",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=["Answer concisely."],
)

# Agent CÓ tools (dùng search_tools từ ở trên)
agent_with_tools = Agent(
    name="Agent With Tools",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web then answer concisely."],
)

question = "Which website currently ranks #1 for the keyword 'learn SEO' on Google?"

print("=== AGENT KHÔNG CÓ TOOLS ===")
r1 = agent_no_tools.run(question)
print(r1.content)

print("\n=== AGENT CÓ TOOLS ===")
r2 = agent_with_tools.run(question)
print(r2.content)

## Sản phẩm sử dụng search như thế nào

Trong sản phẩm thật, agent **Content Writer** sử dụng DataForSEO search để nghiên cứu chủ đề trước khi viết bài.

Từ `output/backend/agents/content_writer.py`:
```python
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

_tools = [save_article, list_all_articles]
_creds = get_dataforseo_credentials()
if _creds:
    _tools.insert(0, DataForSEOSearchTools(login=_creds[0], password=_creds[1]))
```

Chú ý pattern: search là **có điều kiện** (conditional). Nếu `DATA_FOR_SEO_API_KEY` chưa được cài, Content Writer vẫn hoạt động — chỉ là viết từ kiến thức sẵn có thay vì tìm kiếm trước. Sản phẩm vẫn chạy được dù thiếu API key (fallback an toàn).

## Tổng kết bài học 9

Những gì bạn đã học:
- **Tools** giúp agent **hành động**, không chỉ suy nghĩ
- **DataForSEOSearchTools** — tìm kiếm web chuyên nghiệp qua DataForSEO API
- Tools cần **credentials** — `get_dataforseo_credentials()` đọc từ `.env`
- Agent **tự quyết định** khi nào cần dùng tools
- Agent có tools cho kết quả **chính xác và cập nhật hơn**
- Tools có thể là **có điều kiện** (conditional) — sản phẩm vẫn chạy dù có hay không có search

**Bài tiếp theo:** Chúng ta sẽ học cách khiến agent trả về **dữ liệu có cấu trúc** (structured output) — thay vì văn bản tự do.

## Bài tập

Tạo một agent có DataForSEO search tools và instructions để nó nghiên cứu về **công ty của bạn** hoặc một đối thủ cạnh tranh. Hãy hỏi nó một câu cụ thể như:
- "Công ty [tên] cung cấp những dịch vụ gì?"
- "Tìm tin tức gần đây về [công ty]"

In kết quả ra và kiểm tra xem thông tin có chính xác không.

In [ ]:
# Bài tập: Điền vào chỗ trống để tạo một agent nghiên cứu

from agno.agent import Agent
from agno.models.anthropic import Claude

# Tạo agent có khả năng tìm kiếm web
company_researcher = Agent(
    name=___,                                    # Đặt tên cho agent
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=___,                                   # Dùng search_tools ở trên
    instructions=[___],                          # Yêu cầu agent nghiên cứu và tóm tắt
)

# Yêu cầu agent nghiên cứu một công ty
response = company_researcher.run(___)           # Hỏi về một công ty
print(response.content)

<details>
<summary>Bấm để xem đáp án</summary>

```python
from agno.agent import Agent
from agno.models.anthropic import Claude

company_researcher = Agent(
    name="Company Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web for company information and summarize your findings clearly."],
)

response = company_researcher.run("What services does SEONGON offer? Find their website and key information.")
print(response.content)
```
</details>